# Estadística descriptiva y distribuciones

**Sesión 06 · Estadística I**

La estadística es la disciplina que usa **muestras** de datos para respaldar afirmaciones sobre **poblaciones**:

+ una **población** es el conjunto completo de individuos sobre el que queremos saber algo;
+ una **muestra** es la parte de la población que realmente observamos.

Antes de hacer afirmaciones sobre la población (eso es la **inferencia**, en la sesión 07) hay que **describir bien la muestra**:

+ **resumir** los datos con unos pocos números de centro y de dispersión;
+ **representar su distribución**, es decir, con qué frecuencia aparece cada valor;
+ **comparar grupos** sin dejarnos engañar por un único resumen;
+ **modelar** los datos con distribuciones de probabilidad conocidas;
+ **medir la relación** entre dos variables.

Todo lo que calcularemos hoy son **estadísticos descriptivos** de *nuestra* muestra. Si las diferencias que encontremos son reales o fruto del azar es la pregunta de la próxima sesión.

**Contenido**

1. Punto de partida: ¿los primeros hijos nacen más tarde?
2. Los datos: la NSFG
3. Resumir los datos: centro y dispersión
4. La distribución: histograma, PMF y CDF
5. Valores atípicos y asimetría
6. Ejercicio: la carrera Cherry Blossom
7. Riesgo relativo
8. Distribuciones de probabilidad
9. Relación entre dos variables
10. Conclusiones

## 1. Punto de partida: ¿los primeros hijos nacen más tarde?

<center><img src="images/taft_puck_eggdig.jpg"></center>
<center><img src="images/eggs.png"></center>
<center><small>Titular del <i>Chicago Tribune</i>, 13 de junio de 1897.</small></center>

Un titular construido a partir de un solo caso es un ejemplo de **evidencia anecdótica**. Hoy nos hacemos una pregunta que mucha gente da por cierta: **¿los primeros hijos nacen más tarde que los demás?**

Sin un análisis de datos que la respalde, la afirmación también es anecdótica:

+ se basa en **pocos casos**: la experiencia propia, la de amistades y familiares;
+ tiene **sesgo de selección**: a quien más le interesa la afirmación es a quien tuvo un primer hijo que llegó tarde;
+ tiene **sesgo de confirmación**: quien se lo cree tiende a recordar y a contar los casos que la confirman;
+ las fuentes son **poco precisas**: los recuerdos se deforman con el tiempo.

Para salir de la anécdota necesitamos datos. El caso viene de *Think Stats*, de Allen B. Downey, y continúa en la sesión 07.

## 2. Los datos: la NSFG

Desde 1973, los Centros para el Control y la Prevención de Enfermedades (CDC) de EE. UU. realizan la *National Survey of Family Growth* (NSFG), una encuesta sobre vida familiar, matrimonio y divorcio, embarazo, infertilidad, anticoncepción y salud. Usamos el fichero de embarazos del ciclo de 2002, `2002FemPreg.dat`: **un registro por cada embarazo** que declaró una encuestada.

Es un fichero de **ancho fijo**: cada variable ocupa siempre las mismas posiciones de la línea. Leemos solo las que necesitamos:

| Variable | Posiciones | Qué es |
| --- | --- | --- |
| `caseid` | 0–11 | Identificador de la encuestada |
| `birthwgt_lb`, `birthwgt_oz` | 56–57, 58–59 | Peso al nacer, en libras y onzas |
| `prglength` | 274–275 | Duración del embarazo en semanas (en la NSFG se llama `prglngth`) |
| `outcome` | 276 | Resultado del embarazo (1 = nacido vivo) |
| `birthord` | 277–278 | Orden de nacimiento, solo para los nacidos vivos |
| `agepreg` | 283–286 | Edad de la madre al final del embarazo, en centésimas de año |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from scipy import special, stats

plt.style.use('ggplot')
rng = np.random.default_rng(2026)   # semilla fija: los resultados son reproducibles

DATOS = Path('files')
assert (DATOS / '2002FemPreg.dat').exists(), 'Faltan los datos: copia 2002FemPreg.dat y run10.txt en la carpeta files/'

In [ ]:
# Posiciones de cada variable en la línea, contando desde 0 como en Python
COLUMNAS = {
    'caseid': (0, 12), 'birthwgt_lb': (56, 58), 'birthwgt_oz': (58, 60), 'prglength': (274, 276),
    'outcome': (276, 277), 'birthord': (277, 279), 'agepreg': (283, 287),
}
nsfg = pd.read_fwf(DATOS / '2002FemPreg.dat', colspecs=list(COLUMNAS.values()),
                   names=list(COLUMNAS), header=None).fillna({'birthord': 0})

print(f'{len(nsfg)} embarazos')
nsfg.head()

Las posiciones vacías de la línea se leen como valores ausentes (`NaN`). En `birthord` el hueco significa que el embarazo no terminó en un nacimiento vivo, así que lo sustituimos por 0. Veamos cómo terminaron los embarazos:

In [ ]:
nsfg['outcome'].value_counts().sort_index()

Solo 9.148 de los 13.593 embarazos terminaron en un nacimiento vivo (`outcome == 1`). Nos quedamos con ellos y los separamos en dos grupos, primeros hijos y demás hijos, con los mismos nombres que usaremos en la sesión 07:

In [ ]:
nacidos_vivos = nsfg[nsfg.outcome == 1]                                     # outcome == 1: nacido vivo
primeros = nacidos_vivos.loc[nacidos_vivos.birthord == 1, 'prglength']      # duración en semanas
otros = nacidos_vivos.loc[nacidos_vivos.birthord >= 2, 'prglength']

print(f'Primeros hijos: {len(primeros)} · Demás hijos: {len(otros)}')

## 3. Resumir los datos: centro y dispersión

### 3.1 La media

Si tenemos $n$ valores $x_i$, la **media muestral** es su suma dividida entre $n$:

$$ \bar{x} = \frac{1}{n} \sum_i x_i $$

Es el resumen más básico de la **tendencia central** de los datos. ¿Hay diferencia entre los dos grupos?

In [ ]:
diferencia_obs = primeros.mean() - otros.mean()

print(f'Media de los primeros hijos: {primeros.mean():.3f} semanas')
print(f'Media de los demás hijos:    {otros.mean():.3f} semanas')
print(f'Diferencia: {diferencia_obs:.3f} semanas ≈ {diferencia_obs * 7:.2f} días ≈ {diferencia_obs * 7 * 24:.0f} horas')

Los primeros hijos nacen, en promedio, unas 13 horas más tarde. Es un primer **efecto aparente** a favor de la hipótesis. Pero una media sola no basta para describir unos datos.

> No confundas la **media muestral** $\bar{x}$, la de nuestros datos, con la **media poblacional** $\mu$, la de todos los nacimientos, que no conocemos. La distancia entre las dos es el tema de la sesión 07.

### 3.2 Varianza y desviación estándar

La **varianza** mide la **dispersión** de los datos: es la media de los cuadrados de las **desviaciones respecto a la media**, $x_i - \bar{x}$:

$$ s^2 = \frac{1}{n-1} \sum_i (x_i - \bar{x})^2 $$

La **desviación estándar** $s$ es su raíz cuadrada. Se usa más que la varianza porque está en las mismas unidades que los datos: semanas, y no semanas al cuadrado.

> **¿Por qué $n-1$ y no $n$?** Dividir entre $n$ subestima un poco la varianza de la población cuando la muestra es pequeña, y dividir entre $n-1$ lo corrige. pandas divide entre $n-1$ por defecto (`ddof=1`) y NumPy, entre $n$ (`ddof=0`). Con miles de datos la diferencia es despreciable, pero conviene saber qué calcula cada función.

In [ ]:
resumen = pd.DataFrame({
    'n': [len(primeros), len(otros)],
    'media (semanas)': [primeros.mean(), otros.mean()],
    'varianza': [primeros.var(), otros.var()],
    'desv. estándar': [primeros.std(), otros.std()],
}, index=['Primeros hijos', 'Demás hijos'])
resumen.round(3)

### 3.3 La mediana: un resumen robusto

La **mediana** es el valor que queda en el centro cuando ordenamos los datos: la mitad de los valores son menores o iguales que ella y la otra mitad, mayores o iguales. A diferencia de la media, apenas cambia con unos pocos valores extremos. Decimos que es un resumen **robusto**.

<center><img src="images/mean_median.gif"></center>

In [ ]:
print(f'Mediana de los primeros hijos: {primeros.median():.0f} semanas')
print(f'Mediana de los demás hijos:    {otros.median():.0f} semanas')

Las dos medianas son 39 semanas: la mediana no ve la diferencia que veía la media. Un ejemplo pequeño muestra por qué pueden ser tan distintas. Nos pasan seis medidas y una de ellas está mal anotada:

In [ ]:
medidas = np.array([-0.441, 1.774, -0.101, -1.138, 2.975, -213.8])   # la última debería ser -2.138
print(f'Media: {medidas.mean():.3f} · Mediana: {np.median(medidas):.3f}')

Un único valor erróneo arrastra la media hasta −35, mientras que la mediana sigue cerca de 0. Ante datos con errores o con colas largas, mira siempre las dos.

### 3.4 Cuantiles y percentiles

Ordenamos los datos y buscamos el valor $x_p$ que deja por debajo (o igual) una fracción $p$ de ellos y por encima el resto, $1-p$. Ese valor es el **cuantil** $p$ o, si $p$ se expresa en tanto por ciento, el **percentil** $100p$. La mediana es el cuantil 0,5, es decir, el percentil 50.

El **resumen de cinco números** (mínimo, primer cuartil $Q_1$ o percentil 25, mediana, tercer cuartil $Q_3$ o percentil 75 y máximo) da una idea rápida de dónde están los datos y de hasta dónde llegan:

In [ ]:
cuantiles = [0, 0.25, 0.5, 0.75, 1]
cinco_numeros = pd.DataFrame({'Primeros hijos': primeros.quantile(cuantiles).to_numpy(),
                              'Demás hijos': otros.quantile(cuantiles).to_numpy()},
                             index=['mínimo', 'Q1', 'mediana', 'Q3', 'máximo'])
cinco_numeros

Hay algo raro: un nacimiento vivo con 0 semanas de embarazo y otros de hasta 50 semanas. Volveremos a ello en la sección 5. Fíjate también en que $Q_3$ es 40 semanas en los primeros hijos y 39 en los demás: la diferencia parece estar en la parte alta de la distribución.

**Ejercicio 1**

Calcula, para cada grupo, el percentil 90 de la duración del embarazo y la proporción de bebés que nacen después de la semana 40. ¿Qué te dicen estas cifras que no dijera la diferencia de medias?

Pista: `quantile(0.9)` y `(serie > 40).mean()`.

In [ ]:
# Escribe aquí tu solución

_Tu respuesta:_

## 4. La distribución: histograma, PMF y CDF

Los resúmenes pueden engañar: datos muy distintos pueden tener la misma media y la misma desviación estándar (lo comprobarás en la sección 9). Por eso hay que mirar la **distribución** de los datos: con qué frecuencia aparece cada valor.

### 4.1 Histograma

El **histograma** cuenta cuántas veces aparece cada valor o, si los agrupamos, cuántos valores caen en cada intervalo (*bin*). Como la duración se mide en semanas enteras, usamos un *bin* por semana:

In [ ]:
bins = np.arange(0, 52) - 0.5      # un bin por semana, centrado en el número entero

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
ax1.hist(primeros, bins=bins)
ax1.set_title(f'Primeros hijos (n = {len(primeros)})')
ax1.set_xlabel('Duración del embarazo (semanas)')
ax1.set_ylabel('Número de nacimientos')
ax2.hist(otros, bins=bins, color='tab:blue')
ax2.set_title(f'Demás hijos (n = {len(otros)})')
ax2.set_xlabel('Duración del embarazo (semanas)')
plt.show()

La mayoría de los partos se concentran entre las semanas 37 y 42, con un pico muy marcado en la 39, y hay una larga **cola a la izquierda**: los partos prematuros. Como los dos grupos tienen tamaños distintos (4.413 y 4.735), comparar las alturas de las barras no es justo.

### 4.2 PMF: frecuencias relativas

Si dividimos cada frecuencia entre $n$, obtenemos la **función de masa de probabilidad** (PMF, *probability mass function*): la proporción de datos que toma cada valor. Las proporciones suman 1, así que permiten comparar grupos de tamaño distinto.

In [ ]:
pmf_primeros = primeros.value_counts(normalize=True).sort_index()
pmf_otros = otros.value_counts(normalize=True).sort_index()

semanas = np.arange(34, 47)                 # ampliamos la zona central
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(semanas - 0.2, pmf_primeros.reindex(semanas, fill_value=0), width=0.4, label='Primeros hijos')
ax.bar(semanas + 0.2, pmf_otros.reindex(semanas, fill_value=0), width=0.4, label='Demás hijos')
ax.set_xticks(semanas)
ax.set_xlabel('Duración del embarazo (semanas)')
ax.set_ylabel('Proporción')
ax.set_title('Los primeros hijos nacen menos en la semana 39 y más en las semanas 41 y 42')
ax.legend()
plt.show()

Para verlo mejor, representamos la diferencia entre las dos PMF, en puntos porcentuales (p. p.):

In [ ]:
diferencia_pmf = 100 * pmf_primeros.sub(pmf_otros, fill_value=0).reindex(semanas, fill_value=0)

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(semanas, diferencia_pmf)
ax.axhline(0, color='black', lw=1)
ax.set_xticks(semanas)
ax.set_xlabel('Duración del embarazo (semanas)')
ax.set_ylabel('Diferencia (p. p.)')
ax.set_title('Primeros hijos − demás hijos: la diferencia está alrededor de la moda')
plt.show()

En la semana 39 hay 6,6 p. p. menos de primeros hijos y en las semanas 41 y 42, 3,4 y 2,1 p. p. más. **La diferencia no está en el centro de la distribución, sino en cómo se reparten los valores alrededor de él.**

> **Cuidado con `hist(density=True)`.** No da exactamente la PMF: divide entre la anchura del *bin* para que el **área** de las barras sume 1. Solo coincide con la PMF cuando cada *bin* mide 1, como aquí.

### 4.3 CDF: proporción acumulada

La **función de distribución acumulada** (CDF, *cumulative distribution function*) da, para cada valor $x$, la proporción de datos menores o iguales que $x$: $CDF(x) = P(X \le x)$. Se obtiene acumulando la PMF **ordenada por valor**. Tiene dos ventajas: no depende de cómo elijamos los *bins* y permite leer directamente los percentiles (la mediana es el valor en el que la CDF llega a 0,5).

In [ ]:
cdf_primeros = pmf_primeros.cumsum()     # la PMF ya está ordenada por semana
cdf_otros = pmf_otros.cumsum()

fig, ax = plt.subplots(figsize=(12, 4))
ax.step(cdf_primeros.index, cdf_primeros, where='post', lw=2, label='Primeros hijos')
ax.step(cdf_otros.index, cdf_otros, where='post', lw=2, label='Demás hijos')
ax.set_xlim(30, 46)
ax.set_xlabel('Duración del embarazo (semanas)')
ax.set_ylabel('Proporción acumulada')
ax.set_title('Desde la semana 39, la CDF de los primeros hijos va por debajo')
ax.legend()
plt.show()

Si la CDF de un grupo va por debajo de la de otro, ese grupo tiene menos valores pequeños, es decir, valores más grandes. Hasta la semana 37 la curva de los primeros hijos va ligeramente por encima (hay algo más de prematuros) y desde la 39, por debajo (hay más partos tardíos).

**Ejercicio 2**

Lee en el gráfico qué proporción de cada grupo ha nacido al terminar la semana 40. Después, escribe una función `cdf(serie, x)` que calcule esa proporción directamente a partir de los datos y comprueba que coincide con lo que has leído.

In [ ]:
# Escribe aquí tu solución

## 5. Valores atípicos y asimetría

### 5.1 Valores atípicos

Un **valor atípico** (*outlier*) es un dato muy alejado de la tendencia central. Puede ser un error de registro o un caso real pero poco frecuente. Dos formas habituales de buscarlos:

+ los valores que están muy lejos de la mediana;
+ los valores con una **puntuación estándar** grande en valor absoluto (más de 2 o 3 desviaciones estándar):

$$ z_i = \frac{x_i - \bar{x}}{s} $$

La puntuación estándar no tiene unidades: dice a cuántas desviaciones estándar de la media está cada dato. Estandarizar no cambia la forma de la distribución: si la variable es asimétrica, $z$ también lo es.

In [ ]:
duracion = nacidos_vivos['prglength']
mediana = duracion.median()
z = (duracion - duracion.mean()) / duracion.std()

print(f'Mediana: {mediana:.0f} semanas')
print(f'Más de 10 semanas por debajo (< {mediana - 10:.0f}): {(duracion < mediana - 10).sum()} nacimientos')
print(f'Más de 6 semanas por encima (> {mediana + 6:.0f}):   {(duracion > mediana + 6).sum()} nacimientos')
print(f'Con |z| > 3:                          {(z.abs() > 3).sum()} nacimientos')

In [ ]:
# Duraciones fuera del intervalo de 29 a 45 semanas y cuántos nacimientos tiene cada una
fuera = duracion[(duracion < mediana - 10) | (duracion > mediana + 6)]
fuera.value_counts().sort_index().to_frame('nacimientos').T

Un nacimiento vivo tras 0 o 4 semanas de embarazo es, con seguridad, un error de registro. Uno de 24 semanas puede ser real: un gran prematuro. Y uno de 50 semanas es muy improbable. **No hay una regla universal**: decidir qué es un error es una decisión del análisis, y hay que explicarla.

Si los consideramos errores, una opción es **recortar** los datos y quedarnos con las duraciones de 29 a 45 semanas (mediana − 10 y mediana + 6, como en *Think Stats*):

In [ ]:
recortados = nacidos_vivos[nacidos_vivos.prglength.between(29, 45)]
primeros_rec = recortados.loc[recortados.birthord == 1, 'prglength']
otros_rec = recortados.loc[recortados.birthord >= 2, 'prglength']
print(f'Se quitan {len(nacidos_vivos) - len(recortados)} nacimientos')

grupos = {('Primeros hijos', 'todos'): primeros, ('Primeros hijos', 'recortados'): primeros_rec,
          ('Demás hijos', 'todos'): otros, ('Demás hijos', 'recortados'): otros_rec}
comparacion = pd.DataFrame({
    'media': [g.mean() for g in grupos.values()],
    'desv. estándar': [g.std() for g in grupos.values()],
    'mediana': [g.median() for g in grupos.values()],
    'mínimo': [g.min() for g in grupos.values()],
    'máximo': [g.max() for g in grupos.values()],
}, index=pd.MultiIndex.from_tuples(grupos.keys()))
comparacion.round(3)

Al recortar, la media sube (los prematuros extremos tiraban de ella hacia abajo), la desviación estándar baja bastante y la mediana no se mueve. La diferencia de medias pasa de 0,078 a 0,104 semanas: **una decisión de limpieza cambia la cifra que comunicaríamos**. En las secciones 5.2 y 7 usamos los datos recortados.

### 5.2 Medir la asimetría

La **asimetría** (*skewness*) mide hacia qué lado se extiende más la distribución:

$$ g_1 = \frac{m_3}{m_2^{3/2}}, \qquad m_k = \frac{1}{n} \sum_i (x_i - \bar{x})^k $$

El numerador es la media de las desviaciones al cubo y el denominador, la varianza (media de las desviaciones al cuadrado) elevada a 3/2. Si $g_1 < 0$, la distribución se extiende más hacia la izquierda; si $g_1 > 0$, hacia la derecha, y si es simétrica, $g_1 \approx 0$.

Como eleva las desviaciones al cubo, $g_1$ es muy sensible a los valores atípicos. Una alternativa más robusta es el **coeficiente de asimetría de Pearson**, basado en la mediana:

$$ g_p = \frac{3\,(\bar{x} - \text{mediana})}{s} $$

Si la media queda por debajo de la mediana, la cola está a la izquierda y $g_p < 0$.

`stats.describe` calcula de una vez el tamaño, el mínimo y el máximo, la media, la varianza, la asimetría $g_1$ y la curtosis:

In [ ]:
print('Primeros hijos:', stats.describe(primeros_rec))
print('Demás hijos:   ', stats.describe(otros_rec))

**Ejercicio 3**

Escribe dos funciones, `asimetria_g1(x)` y `asimetria_pearson(x)`, que calculen $g_1$ y $g_p$. Aplícalas a la duración del embarazo de los dos grupos, con todos los datos y con los recortados, y comprueba que tu $g_1$ coincide con la asimetría de `stats.describe`. ¿Qué coeficiente cambia más al recortar? ¿Por qué?

In [ ]:
# Escribe aquí tu solución

_Tu respuesta:_

**Para pensar**

1. ¿Se te ocurre un ejemplo real en el que todos los datos sean menores o iguales que la media?
2. ¿Y uno en el que todos los datos sean menores o iguales que la mediana? Sería una distribución muy asimétrica hacia la izquierda.
3. Si preguntamos a un grupo de personas «¿cómo conduces en comparación con el conductor medio?», ¿qué forma tendría la distribución de las respuestas?

_Tu respuesta:_

## 6. Ejercicio: la carrera Cherry Blossom

El fichero `run10.txt` contiene los tiempos de los 16.924 corredores que terminaron la carrera Cherry Blossom de 10 millas de 2012, en Washington D. C. (EE. UU.). Está separado por tabuladores y se lee con `pd.read_table`. La columna `time` es el tiempo en minutos y `gender`, el sexo (`F` o `M`).

In [ ]:
carrera = pd.read_table(DATOS / 'run10.txt')
carrera.head()

**Ejercicio 4**

a) Calcula el tiempo medio y el tiempo mediano de toda la carrera.

b) Calcula la diferencia de tiempo medio entre mujeres (`F`) y hombres (`M`).

c) Representa las dos distribuciones con histogramas normalizados en el mismo gráfico. Pon un título que diga la conclusión.

d) ¿Es simétrica la distribución de los tiempos? Usa las funciones del ejercicio 3.

In [ ]:
# Escribe aquí tu solución

_Tu respuesta:_

## 7. Riesgo relativo

A veces interesa más una proporción que una media. Clasificamos los partos en tres categorías:

+ **prematuro**: en la semana 37 o antes;
+ **a término**: en las semanas 38, 39 o 40;
+ **tardío**: en la semana 41 o después.

Calculamos la proporción de cada categoría en los dos grupos, con los datos recortados:

In [ ]:
def proporciones(duracion: pd.Series) -> pd.Series:
    # Proporción de partos prematuros, a término y tardíos
    return pd.Series({
        'prematuro': (duracion <= 37).mean(),
        'a término': duracion.between(38, 40).mean(),
        'tardío': (duracion >= 41).mean(),
    })


tabla_prop = pd.DataFrame({'Primeros hijos': proporciones(primeros_rec),
                           'Demás hijos': proporciones(otros_rec)})
tabla_prop.round(3)

El **riesgo relativo** es el cociente de dos probabilidades. Por ejemplo, la probabilidad de que un primer hijo nazca prematuro es del 17,2 % y la de los demás hijos, del 16,0 %, así que el riesgo relativo de nacer prematuro es 17,2 / 16,0 ≈ 1,07:

In [ ]:
riesgo_relativo = tabla_prop['Primeros hijos'] / tabla_prop['Demás hijos']
riesgo_relativo.round(2)

Un riesgo relativo de 1,07 significa que los primeros hijos tienen un 7 % más de probabilidad de nacer prematuros; uno de 1,67, que tienen un 67 % más de probabilidad de nacer tarde.

**El riesgo relativo es un cociente: no dice si las probabilidades son grandes o pequeñas.** «Un 67 % más» de un 9,4 % es un 15,7 %. Cuando comuniques un riesgo relativo, da también las dos proporciones.

## 8. Distribuciones de probabilidad

Hasta ahora hemos trabajado con **distribuciones empíricas**: las de los valores que hemos observado. Muchos fenómenos se describen bien con **distribuciones teóricas**, definidas por una fórmula y unos pocos parámetros. Sirven para resumir los datos con muy pocos números, para calcular probabilidades y para simular.

### 8.1 La probabilidad como frecuencia

La definición más habitual de **probabilidad** es la de una frecuencia: si en $n$ ensayos idénticos observamos $s$ éxitos, la probabilidad de éxito se aproxima por $s/n$, y la aproximación mejora cuanto mayor es $n$. Es la interpretación **frecuentista**, la que usamos en esta sesión y en la siguiente. La interpretación **bayesiana**, que entiende la probabilidad como un grado de creencia, es el tema de la sesión 08.

Algo de vocabulario. Cada repetición de un experimento aleatorio es un **ensayo**, y lo que puede ocurrir o no en él, un **suceso**. Al tirar un dado, cada tirada es un ensayo; si nos interesa sacar un 6, cada 6 es un **éxito** y el resto, **fracasos**. Dos ensayos son **independientes** si el resultado de uno no cambia la probabilidad del otro, como dos tiradas de un dado. Entonces, $P(A \text{ y } B) = P(A)\,P(B)$.

### 8.2 La distribución binomial

Si repetimos $n$ ensayos independientes con probabilidad de éxito $p$, la probabilidad de obtener exactamente $k$ éxitos es

$$ P(k) = \binom{n}{k}\, p^k (1-p)^{n-k}, \qquad \binom{n}{k} = \frac{n!}{k!\,(n-k)!} $$

donde $\binom{n}{k}$ cuenta de cuántas formas se pueden colocar $k$ éxitos entre $n$ ensayos. Es la **distribución binomial**. En la sesión 07 la usaremos para decidir si una moneda está sesgada.

¿Qué probabilidad hay de obtener 2 éxitos en 5 ensayos con $p = 0{,}5$? ¿Y 5 caras en 9 lanzamientos de una moneda?

In [ ]:
formas = special.comb(5, 2, exact=True)
print(f'Formas de colocar 2 éxitos en 5 ensayos: {formas}')
print(f'P(2 éxitos en 5 ensayos) · fórmula: {formas * 0.5**2 * (1 - 0.5)**3:.4f} · scipy: {stats.binom.pmf(2, 5, 0.5):.4f}')
print(f'P(5 caras en 9 lanzamientos):         {stats.binom.pmf(5, 9, 0.5):.4f}')

k = np.arange(0, 10)
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(k, stats.binom.pmf(k, 9, 0.5))
ax.set_xticks(k)
ax.set_xlabel('Número de caras en 9 lanzamientos')
ax.set_ylabel('Probabilidad')
ax.set_title('Distribución binomial (n = 9, p = 0,5)')
plt.show()

**Ejercicio 5**

¿Qué probabilidad hay de sacar exactamente 6 seises en 9 tiradas de un dado? Calcúlala con la fórmula y con `stats.binom.pmf`.

In [ ]:
# Escribe aquí tu solución

### 8.3 Simulación de Monte Carlo

Los **experimentos de Monte Carlo** calculan una probabilidad **simulando** el experimento muchas veces y contando con qué frecuencia ocurre el suceso. Son útiles cuando no hay fórmula o es difícil de obtener, y son la base de muchos métodos de la sesión 07, como el *bootstrap* y los tests de permutación.

Empezamos por un caso trivial: la probabilidad de sacar un 6 en una tirada.

In [ ]:
tiradas = rng.integers(1, 7, size=1_000_000)          # un millón de tiradas de un dado
print(f'Proporción de seises: {np.mean(tiradas == 6):.4f} (teórica: 1/6 = {1/6:.4f})')

¿Y la probabilidad de sacar exactamente un 6 en dos tiradas? La binomial da $\binom{2}{1} \cdot \frac{1}{6} \cdot \frac{5}{6} \approx 0{,}278$. Lo comprobamos simulando 10.000 experimentos de dos tiradas:

In [ ]:
dos_tiradas = rng.integers(1, 7, size=(10_000, 2))     # cada fila es un experimento
un_seis = (dos_tiradas == 6).sum(axis=1) == 1
print(f'Exactamente un 6 en dos tiradas · simulación: {un_seis.mean():.4f} · binomial: {stats.binom.pmf(1, 2, 1/6):.4f}')

**Ejercicio 6**

Tiras dos dados, uno negro y uno rojo. ¿Qué probabilidad hay de que el negro saque más puntos que el rojo? Calcúlala con una simulación y compruébala contando los casos posibles.

In [ ]:
# Escribe aquí tu solución

**Un caso más interesante.** Si tiras un dado 100 veces, ¿qué probabilidad hay de sacar al menos seis seises seguidos? Aquí la fórmula no es sencilla, pero simularlo sí lo es:

In [ ]:
def hay_racha(exitos: np.ndarray, longitud: int) -> np.ndarray:
    # Para cada fila (un experimento), True si contiene al menos `longitud` éxitos seguidos
    ventanas = sliding_window_view(exitos, longitud, axis=1)   # todos los tramos consecutivos de `longitud` ensayos
    return ventanas.all(axis=2).any(axis=1)


n_exp = 100_000
seises = rng.integers(1, 7, size=(n_exp, 100)) == 6       # 100 000 experimentos de 100 tiradas
print(f'P(al menos 6 seises seguidos en 100 tiradas) ≈ {hay_racha(seises, 6).mean():.4f}')

**Ejercicio 7 (opcional)**

¿Qué probabilidad hay de que un futbolista marque en al menos 10 partidos seguidos de una temporada de 42 partidos? Supón que cada partido es un ensayo independiente y, para simplificar, que la probabilidad de marcar en un partido es igual a su media de goles por partido. Según datos de 2013, Messi llevaba 323 goles en 387 partidos (0,83 por partido) y Cristiano Ronaldo, 329 goles en 527 partidos (0,62 por partido).

Pista: `rng.random((n_exp, 42)) < p` simula si marca o no en cada partido, y `hay_racha` hace el resto.

Para terminar, piensa en la simplificación: ¿es lo mismo marcar 0,83 goles por partido que marcar en el 83 % de los partidos? ¿La probabilidad real sería mayor o menor que la que has calculado?

In [ ]:
# Escribe aquí tu solución

_Tu respuesta:_

### 8.4 Distribuciones continuas: la exponencial

Para variables continuas, como tiempos, pesos o distancias, la PMF se sustituye por una **función de densidad** (PDF, *probability density function*): la probabilidad de que la variable caiga en un intervalo es el **área** bajo la PDF en ese intervalo. La CDF se define igual que antes, $CDF(x) = P(X \le x)$.

La **distribución exponencial** tiene un único parámetro, la tasa $\lambda$:

$$ PDF(x) = \lambda\, e^{-\lambda x}, \qquad CDF(x) = 1 - e^{-\lambda x}, \qquad x \ge 0 $$

Su media es $1/\lambda$, su varianza, $1/\lambda^2$, y su mediana, $\ln(2)/\lambda$.

Aparece al medir el **tiempo entre sucesos** que ocurren al azar con una tasa constante. Por ejemplo, en un hospital australiano se registraron 44 nacimientos en 24 horas, es decir, una tasa de $\lambda = 44/1440 \approx 0{,}0306$ nacimientos por minuto. La figura muestra la CDF de los tiempos entre nacimientos observados:

<center><img src="images/interarrivaltimes.png"></center>

In [ ]:
lam = 44 / (24 * 60)              # nacimientos por minuto
x = np.linspace(0, 160, 300)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(x, lam * np.exp(-lam * x))
ax1.set_title(f'PDF exponencial (λ = {lam:.4f})')
ax1.set_xlabel('Minutos hasta el siguiente nacimiento')
ax1.set_ylabel('Densidad')
ax2.plot(x, 1 - np.exp(-lam * x))
ax2.axvline(np.log(2) / lam, color='black', ls='--', lw=1, label='Mediana')
ax2.set_title(f'CDF exponencial (λ = {lam:.4f})')
ax2.set_xlabel('Minutos hasta el siguiente nacimiento')
ax2.set_ylabel('Proporción acumulada')
ax2.legend()
plt.show()

print(f'Tiempo medio entre nacimientos: {1 / lam:.1f} min · mediana: {np.log(2) / lam:.1f} min')

En la realidad, una tasa constante casi nunca se cumple del todo: las llamadas a un servicio de atención no llegan igual a las 4 de la madrugada que a las 11 de la mañana. Pero en un intervalo en el que la tasa es más o menos estable (de 14 a 16 h en días laborables, por ejemplo), la exponencial es una buena aproximación del tiempo hasta la siguiente llamada. Lo mismo ocurre con el tiempo hasta que se desintegra una partícula radiactiva o con el tiempo hasta el impago de una deuda en algunos modelos de riesgo de crédito.

**Ejemplo resuelto.** El número de días de antelación con que los viajeros compran un billete de avión sigue una distribución exponencial con media de 15 días. (a) ¿Qué probabilidad hay de que un viajero compre el billete con menos de 10 días de antelación? (b) ¿Cuántos días de antelación o menos tiene la mitad de los viajeros?

Como la media es $1/\lambda = 15$, la tasa es $\lambda = 1/15$. Entonces, $P(X < 10) = 1 - e^{-10/15}$, y la mitad de los viajeros compra con una antelación igual o menor que la mediana, $\ln(2)/\lambda$. En `scipy.stats` la exponencial se define con `scale` $= 1/\lambda$:

In [ ]:
lam_billetes = 1 / 15
print(f'(a) P(X < 10) = {1 - np.exp(-lam_billetes * 10):.4f} · scipy: {stats.expon.cdf(10, scale=15):.4f}')
print(f'(b) Mediana = {np.log(2) / lam_billetes:.1f} días · scipy: {stats.expon.median(scale=15):.1f} días')

**Ejercicio 8**

Un dispositivo dura, de media, 10 años, y su duración sigue una distribución exponencial.

a) ¿Qué probabilidad hay de que dure más de 7 años?

b) Si se usan cinco dispositivos, uno detrás de otro, ¿cuánto durarán en total, de media?

c) ¿Qué probabilidad hay de que un dispositivo dure entre 9 y 11 años?

In [ ]:
# Escribe aquí tu solución

### 8.5 La distribución normal

La **distribución normal** o **gaussiana** es la más usada, porque describe muchos fenómenos y porque es cómoda para el análisis. Tiene dos parámetros, la media $\mu$ y la desviación estándar $\sigma$, y su PDF es la conocida campana:

$$ PDF(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right) $$

Su CDF no tiene una expresión cerrada, pero `scipy.stats` la calcula (`stats.norm.cdf`).

In [ ]:
mu, sigma = 5, 1
x = np.linspace(0, 10, 300)
pdf_formula = np.exp(-(x - mu)**2 / (2 * sigma**2)) / np.sqrt(2 * np.pi * sigma**2)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, pdf_formula, lw=6, alpha=0.4, label='Fórmula')
ax.plot(x, stats.norm.pdf(x, mu, sigma), color='black', ls='--', label='stats.norm.pdf')
ax.set_xlabel('x')
ax.set_ylabel('Densidad')
ax.set_title(f'PDF normal (μ = {mu}, σ = {sigma})')
ax.legend()
plt.show()

Siguen aproximadamente una distribución normal muchas medidas de los seres vivos: talla, longitud de partes que crecen (pelo, uñas, dientes) o la presión arterial de las personas adultas. Una razón es que muchas de estas variables son la **suma de muchos efectos pequeños**, y las sumas tienden a ser normales. Es el **teorema central del límite**, que veremos en la sesión 07.

Hay muchas otras distribuciones. Por ejemplo, la de **Pareto** describe la riqueza, el tamaño de las ciudades o de los incendios forestales, y la **lognormal**, el peso de las personas adultas.


## 9. Relación entre dos variables

Hasta ahora hemos descrito las variables de una en una. Ahora nos preguntamos si dos variables **varían juntas**: ¿pesan más al nacer los bebés de embarazos más largos?

Calculamos el peso en kilos a partir de las libras y las onzas (1 libra = 16 onzas = 0,4536 kg). Antes hay que limpiar: en la NSFG, los valores 97, 98 y 99 son **códigos** («no consta», «no contesta», «no sabe»), no pesos, y hay un 51 en las libras que es un error de registro. Por eso solo aceptamos libras por debajo de 20 y onzas por debajo de 16. Aquí usamos todos los nacidos vivos con peso.

In [ ]:
LIBRA_KG = 0.45359237
bebes = nacidos_vivos.assign(
    peso_kg=lambda d: (d['birthwgt_lb'].where(d['birthwgt_lb'] < 20)      # los códigos 97-99 pasan a NaN
                       + d['birthwgt_oz'].where(d['birthwgt_oz'] < 16) / 16) * LIBRA_KG,
    edad_madre=lambda d: d['agepreg'] / 100,                              # agepreg está en centésimas de año
).dropna(subset=['peso_kg'])
print(f'{len(bebes)} bebés con peso')

semanas_jitter = bebes['prglength'] + rng.uniform(-0.3, 0.3, size=len(bebes))   # separa los puntos de cada semana
fig, ax = plt.subplots(figsize=(11, 5))
ax.scatter(semanas_jitter, bebes['peso_kg'], s=8, alpha=0.15)
ax.set_xlabel('Duración del embarazo (semanas)')
ax.set_ylabel('Peso al nacer (kg)')
ax.set_title('El peso sube con las semanas hasta la 41 y después se estanca')
plt.show()

### 9.1 Covarianza

Si dos variables $X$ e $Y$ varían juntas, sus desviaciones respecto a la media, $x_i - \bar{x}$ e $y_i - \bar{y}$, tienden a tener el mismo signo. Su producto es positivo cuando tienen el mismo signo y negativo cuando lo tienen distinto. La **covarianza** es la media de esos productos:

$$ Cov(X, Y) = \frac{1}{n} \sum_i (x_i - \bar{x})(y_i - \bar{y}) $$

Fíjate en que $Cov(X, X)$ es la varianza de $X$.

In [ ]:
def covarianza(x, y) -> float:
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    return np.mean((x - x.mean()) * (y - y.mean()))


X = [5, -1, 3.3, 2.7, 12.2]
Y = [10, 12, 8, 9, 11]
print(f'Cov(X, X) = {covarianza(X, X):.2f} · Var(X) = {np.var(X):.2f}')
print(f'Cov(X, Y) = {covarianza(X, Y):.2f}')
print(f"Cov(semanas, peso) = {covarianza(bebes['prglength'], bebes['peso_kg']):.3f} semanas·kg")

La covarianza entre las semanas y el peso es positiva, pero ¿0,85 semanas·kg es mucho o poco? Es difícil de interpretar, porque depende de las unidades: en gramos sería mil veces mayor.

### 9.2 Correlación de Pearson

La solución es trabajar con las puntuaciones estándar de la sección 5.1: dividimos cada desviación entre la desviación estándar y calculamos la media de los productos. Es el **coeficiente de correlación de Pearson**:

$$ \rho = \frac{1}{n} \sum_i \frac{x_i - \bar{x}}{\sigma_X} \cdot \frac{y_i - \bar{y}}{\sigma_Y} = \frac{Cov(X, Y)}{\sigma_X\, \sigma_Y} $$

No tiene unidades y está siempre entre −1 y 1: vale 1 si los puntos están sobre una recta creciente, −1 si están sobre una recta decreciente y 0 si no hay relación lineal.

In [ ]:
def pearson(x, y) -> float:
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    return covarianza(x, y) / (x.std() * y.std())     # std de NumPy (ddof=0), coherente con covarianza()


print(f"Pearson (nuestra función): {pearson(bebes['prglength'], bebes['peso_kg']):.3f}")
print(f"Pearson (pandas .corr):    {bebes['prglength'].corr(bebes['peso_kg']):.3f}")

<center><img src="images/pearson.png"></center>

Que $\rho = 0$ **no** significa que no haya relación entre las variables: Pearson solo mide relaciones **lineales**. La fila de abajo de la figura tiene relaciones muy claras con $\rho = 0$.

### 9.3 Correlación de rangos de Spearman

Pearson funciona bien si la relación es lineal y las variables son aproximadamente normales, pero no es robusta ante valores atípicos. La **correlación de Spearman** lo mitiga: sustituye cada valor por su **rango**, es decir, su posición en la muestra ordenada, y calcula Pearson sobre los rangos. Por ejemplo, en la muestra {7, 1, 2, 5}, el rango del 5 es 3, porque es el tercero al ordenarla. Si hay empates, cada valor empatado recibe el rango medio.

In [ ]:
def spearman(x, y) -> float:
    # Pearson sobre los rangos; pandas da el rango medio a los empates
    return pearson(pd.Series(x).rank(), pd.Series(y).rank())


print('Rangos de [7, 1, 2, 5]:', pd.Series([7, 1, 2, 5]).rank().tolist())

X = [1, 2, 3, 4, 100]
Y = [5, -100, 7, 10, 9]
print(f'Datos con atípicos · Pearson: {pearson(X, Y):.2f} · Spearman: {spearman(X, Y):.2f}')
print(f"Semanas y peso      · Pearson: {pearson(bebes['prglength'], bebes['peso_kg']):.3f} · "
      f"Spearman: {spearman(bebes['prglength'], bebes['peso_kg']):.3f} · "
      f"pandas: {bebes['prglength'].corr(bebes['peso_kg'], method='spearman'):.3f}")

Entre las semanas y el peso, Pearson (0,49) sale más alta que Spearman (0,36): unos pocos bebés muy prematuros y muy ligeros, en la esquina inferior izquierda del gráfico, forman una relación lineal muy marcada que pesa mucho en Pearson. Spearman solo mira el orden y los tiene menos en cuenta.

> **Correlación no es causalidad.** Que dos variables varíen juntas no dice cuál influye en cuál, ni si las dos dependen de una tercera. Lo trabajaremos en la sesión de inferencia causal.

**Ejercicio 9**

a) Calcula Pearson y Spearman entre la edad de la madre (`edad_madre`) y el peso al nacer. ¿Hay relación?

b) Repite el cálculo entre las semanas y el peso solo con los embarazos de 29 a 45 semanas. ¿Qué coeficiente cambia más? ¿Por qué?

In [ ]:
# Escribe aquí tu solución

_Tu respuesta:_

### 9.4 Mira los datos antes que el número

Un coeficiente de correlación es un modelo muy simple: resume la relación entre dos variables con un solo número, como si los puntos estuvieran alrededor de una recta. Abrir la caja es mirar qué datos producen ese número.

En 1973, el estadístico Francis Anscombe construyó cuatro conjuntos de 11 puntos, conocidos como el **cuarteto de Anscombe**:

In [ ]:
x_comun = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
anscombe = {
    'I': (x_comun, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    'II': (x_comun, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    'III': (x_comun, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    'IV': ([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}

**Ejercicio 10**

a) Calcula, para cada conjunto, la media y la varianza de $x$ y de $y$, la covarianza, la correlación de Pearson y la de Spearman. Reúne los resultados en una tabla.

b) Dibuja los cuatro diagramas de dispersión uno al lado del otro.

c) ¿Qué conclusión sacarías si solo vieras la tabla? ¿Y al ver los gráficos? ¿En qué conjuntos Spearman avisa de que algo es distinto?

In [ ]:
# Escribe aquí tu solución

_Tu respuesta:_

## 10. Conclusiones

### Resumen

+ Antes de resumir, **mira la distribución**: el histograma, la PMF para comparar grupos de distinto tamaño y la CDF para leer percentiles sin depender de los *bins*.
+ La media y la desviación estándar son sensibles a los valores atípicos; la mediana y los percentiles son **robustos**. Para describir una distribución asimétrica, da los dos tipos de resumen.
+ Decidir qué es un valor atípico, y qué hacer con él, es una **decisión del análisis** que cambia las cifras: hay que justificarla y explicarla.
+ Para comparar proporciones, el **riesgo relativo**, acompañado siempre de las dos proporciones.
+ Las **distribuciones teóricas** (binomial, exponencial, normal) describen muchos fenómenos con pocos parámetros. Cuando no hay fórmula, **simulamos** (Monte Carlo).
+ La **covarianza**, la **correlación de Pearson** y la **de Spearman** miden si dos variables varían juntas. Pearson solo ve relaciones lineales y es sensible a los atípicos; ningún coeficiente sustituye al gráfico.

### Efectos aparentes

Después de explorar los datos, hemos encontrado varios **efectos aparentes** a favor de la hipótesis inicial:

+ **Medias.** Con los datos recortados, los primeros hijos nacen de media a las 38,76 semanas y los demás, a las 38,66: unas 17 horas de diferencia.
+ **Distribución.** Menos primeros hijos nacen en la semana 39 (47,9 % frente a 54,5 %) y más en las semanas 41 y 42.
+ **Riesgo relativo.** Los primeros hijos tienen un 67 % más de probabilidad de nacer tarde (15,7 % frente a 9,4 %).

Todo son descriptivos de **una** muestra. Con otras mujeres encuestadas, las cifras cambiarían. ¿Es una diferencia de 0,1 semanas mayor de lo que cabría esperar por azar? ¿Es relevante? Son las preguntas de la sesión 07.

### Para ir más allá (opcional)

También podemos comparar los primeros hijos con los demás **de la misma mujer** (`caseid`). Aunque parezca poco probable, podría ser que en cada mujer el segundo, el tercer… hijo tendiera a nacer antes que el primero. Calcula, para las mujeres con al menos dos hijos nacidos vivos, la diferencia media entre la duración del embarazo del primer hijo y la del segundo.


### Referencia

Allen B. Downey, [*Think Stats: Probability and Statistics for Programmers*](https://greenteapress.com/wp/think-stats-2e/), O'Reilly Media.